# Index Setup and Ingestion

**Notebook 1 of 4.** Creates the `arxiv-nlp` vector + semantic index on Azure AI Search,
generates embeddings for 3,000 NLP paper abstracts via the APIM gateway, and uploads
the documents.

Run this notebook **once** before the others. All subsequent notebooks depend on this index.

**What this notebook does:**
1. Creates the `arxiv-nlp` index with HNSW vector fields (3,072 dims), a `group_ids`
   security-trimming field, an integrated vectorizer wired to the APIM gateway, and
   a semantic configuration
2. Downloads 3,000 NLP paper abstracts from `MaartenGr/arxiv_nlp` (HuggingFace)
3. Generates embeddings via `text-embedding-3-large` through the APIM gateway
4. Uploads all documents to the index

> **Hub/spoke alignment** — no model deployments exist in the IQ spoke resource group.
> All embedding calls go through the APIM gateway. The integrated vectorizer is
> configured with the APIM URL and an API key so query-time embedding also routes
> through APIM.

## Prerequisites

1. **Run `10-01-deploy-search-and-project.ipynb`** — provisions the AI Search service and `iq-project`,
   and writes `IQ_SEARCH_ENDPOINT`, `IQ_GATEWAY_KEY`, and related keys to `.env`.
2. **Python environment** — run `uv sync` from the repo root; select the `.venv` kernel.
3. **Azure CLI** — run `az login` before executing cells.
4. **RBAC** — your identity needs **Search Index Data Contributor** and **Search Service
   Contributor** on the AI Search service (granted by the Bicep in 10-01-deploy-search-and-project).

## 1. Imports and configuration

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    HnswAlgorithmConfiguration,
    HnswParameters,
    SearchableField,
    SearchField,
    SearchFieldDataType,
    SearchIndex,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
    SimpleField,
    VectorSearch,
    VectorSearchProfile,
)
from openai import AzureOpenAI

repo_root = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path.cwd().parent
load_dotenv(repo_root / '.env', override=True)

INDEX_NAME       = 'arxiv-nlp'
search_endpoint  = os.environ['IQ_SEARCH_ENDPOINT']
gateway_url      = os.environ['GATEWAY_URL']
iq_gateway_key   = os.environ['IQ_GATEWAY_KEY']
embedding_model  = os.environ.get('EMBEDDING_MODEL', 'text-embedding-3-large')
VECTOR_DIMS      = 3072  # text-embedding-3-large

# AzureOpenAI SDK appends /openai/ to azure_endpoint automatically.
# GATEWAY_URL already ends with /openai, so strip it to avoid double-prefix.
apim_base = gateway_url.rstrip('/').removesuffix('/openai')

print(f'Search endpoint : {search_endpoint}')
print(f'Gateway URL     : {gateway_url}')
print(f'APIM base       : {apim_base}')
print(f'Embedding model : {embedding_model}')
print(f'Vector dims     : {VECTOR_DIMS}')
print(f'Index name      : {INDEX_NAME}')

Search endpoint : https://iq-search-n5d3ja.search.windows.net
Gateway URL     : https://apim-foundry-c2676f.azure-api.net/openai
APIM base       : https://apim-foundry-c2676f.azure-api.net
Embedding model : text-embedding-3-large
Vector dims     : 3072
Index name      : arxiv-nlp


## 2. Create clients

`DefaultAzureCredential` handles all search operations. RBAC assignments granted by
the Bicep deployment give the deployer identity **Search Index Data Contributor** and
**Search Service Contributor** — no admin key required.

In [2]:
credential = DefaultAzureCredential()

index_client = SearchIndexClient(
    endpoint=search_endpoint,
    credential=credential,
)
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=INDEX_NAME,
    credential=credential,
)

# AzureOpenAI client pointed at the APIM gateway — all embeddings go through here.
# Use apim_base (without /openai suffix) as azure_endpoint; the SDK appends /openai/ itself.
embed_client = AzureOpenAI(
    azure_endpoint=apim_base,
    api_key=iq_gateway_key,
    api_version='2024-10-21',
)

print('Search index client : ready')
print('Search client       : ready')
print('Embed client        : ready (APIM gateway)')

Search index client : ready
Search client       : ready
Embed client        : ready (APIM gateway)


## 3. Create the search index

The `arxiv-nlp` index schema:

| Field | Type | Notes |
|-------|------|-------|
| `id` | String (key) | Filterable |
| `title` | String (searchable) | Filterable, sortable |
| `abstract` | String (searchable) | `en.microsoft` analyser |
| `year` | Int32 | Filterable, sortable |
| `categories` | String (searchable) | Filterable, facetable |
| `group_ids` | Collection(String) | Security-trimming field — documents are tagged with group IDs at ingest; queries can filter by `group_ids/any(...)` to enforce per-document access control |
| `titleVector` | Collection(Single) | 3,072 dims, HNSW cosine, `stored=False` |
| `abstractVector` | Collection(Single) | 3,072 dims, HNSW cosine, `stored=False` |

**Integrated vectorizer** — the search service calls `text-embedding-3-large` via the
APIM gateway at query time. The vectorizer is configured with the APIM URL and API key,
so the search service's managed identity is not involved in embedding calls.

> `stored=False` on vector fields saves storage costs — vectors are used only for ANN
> search and are not returned in query results.

In [3]:
fields = [
    SimpleField(
        name='id',
        type=SearchFieldDataType.String,
        key=True,
        filterable=True,
    ),
    SearchableField(
        name='title',
        type=SearchFieldDataType.String,
        filterable=True,
        sortable=True,
    ),
    SearchableField(
        name='abstract',
        type=SearchFieldDataType.String,
        analyzer_name='en.microsoft',
    ),
    SimpleField(
        name='year',
        type=SearchFieldDataType.Int32,
        filterable=True,
        sortable=True,
    ),
    SearchableField(
        name='categories',
        type=SearchFieldDataType.String,
        filterable=True,
        facetable=True,
    ),
    # Security-trimming field — OData filter: group_ids/any(g: search.in(g, 'admins,researchers'))
    SearchField(
        name='group_ids',
        type=SearchFieldDataType.Collection(SearchFieldDataType.String),
        filterable=True,
    ),
    SearchField(
        name='titleVector',
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        stored=False,
        vector_search_dimensions=VECTOR_DIMS,
        vector_search_profile_name='arxiv-nlp-hnsw-profile',
    ),
    SearchField(
        name='abstractVector',
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        stored=False,
        vector_search_dimensions=VECTOR_DIMS,
        vector_search_profile_name='arxiv-nlp-hnsw-profile',
    ),
]

vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name='arxiv-nlp-hnsw',
            parameters=HnswParameters(
                metric='cosine',
                m=4,
                ef_construction=400,
                ef_search=500,
            ),
        )
    ],
    profiles=[
        VectorSearchProfile(
            name='arxiv-nlp-hnsw-profile',
            algorithm_configuration_name='arxiv-nlp-hnsw',
            vectorizer_name='arxiv-nlp-vectorizer',
        )
    ],
    vectorizers=[
        # Integrated vectorizer calls APIM at query time using an API key.
        # resource_url must be the root APIM URL (without /openai); the search
        # service appends /openai/deployments/{model}/embeddings itself.
        AzureOpenAIVectorizer(
            vectorizer_name='arxiv-nlp-vectorizer',
            parameters=AzureOpenAIVectorizerParameters(
                resource_url=apim_base,
                deployment_name=embedding_model,
                model_name=embedding_model,
                api_key=iq_gateway_key,
            ),
        )
    ],
)

semantic_search = SemanticSearch(
    configurations=[
        SemanticConfiguration(
            name='arxiv-nlp-semantic',
            prioritized_fields=SemanticPrioritizedFields(
                title_field=SemanticField(field_name='title'),
                content_fields=[SemanticField(field_name='abstract')],
                keywords_fields=[SemanticField(field_name='categories')],
            ),
        )
    ]
)

index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)

result = index_client.create_or_update_index(index)
print(f"Index '{result.name}' created/updated ({len(result.fields)} fields)")

Index 'arxiv-nlp' created/updated (8 fields)


## 4. Prepare the dataset *(one-time)*

Downloads 3,000 rows from [MaartenGr/arxiv_nlp](https://huggingface.co/datasets/MaartenGr/arxiv_nlp)
(MIT licence) and saves them to `assets/arxiv_nlp_3000.csv` in this directory.

The file is committed after the first run — subsequent runs skip the download.

> The 3,000-row subset spans 1994–2024 across NLP research. This creates genuine
> retrieval challenges (vocabulary diversity, temporal range) that demonstrate the
> value of different search patterns in 10-04-search-patterns.

In [4]:
assets_dir = Path('../assets')
assets_dir.mkdir(exist_ok=True)
csv_path = assets_dir / 'arxiv_nlp_3000.csv'

if csv_path.exists():
    print(f'Dataset already exists at {csv_path} — skipping download.')
else:
    print('Downloading MaartenGr/arxiv_nlp from HuggingFace...')
    from datasets import load_dataset
    ds = load_dataset('MaartenGr/arxiv_nlp', split='train')
    _df = ds.to_pandas().head(3000)
    _df.to_csv(csv_path, index=False)
    size_mb = csv_path.stat().st_size / 1024 / 1024
    print(f'Saved {len(_df)} rows to {csv_path} ({size_mb:.1f} MB)')

print(f'Path: {csv_path.resolve()}')

Dataset already exists at ../assets/arxiv_nlp_3000.csv — skipping download.
Path: /home/jp/development/corticalstack/foundry-nextgen/assets/arxiv_nlp_3000.csv


## 5. Load and preview the dataset

In [5]:
MAX_DOCS = None  # set to an integer (e.g. 50) to test with fewer documents

assert csv_path.exists(), f'Dataset not found at {csv_path} — run section 4 first'

df = pd.read_csv(csv_path)
df = df.rename(columns={
    'Titles':     'title',
    'Abstracts':  'abstract',
    'Years':      'year',
    'Categories': 'categories',
})

if MAX_DOCS is not None:
    df = df.head(MAX_DOCS)
    print(f'[DEV] Limiting to {MAX_DOCS} documents')

print(f'Shape     : {df.shape}')
print(f'Columns   : {list(df.columns)}')
print(f'Year range: {df["year"].min()} - {df["year"].max()}')
print()
print(df[['title', 'year', 'categories']].head(3).to_string(index=False))

Shape     : (3000, 4)
Columns   : ['title', 'abstract', 'year', 'categories']
Year range: 1994 - 2024

                                                           title  year               categories
Introduction to Arabic Speech Recognition Using CMUSphinx System  2007 Computation and Language
              Arabic Speech Recognition System using CMU-Sphinx4  2007 Computation and Language
       On the Development of Text Input Method - Lessons Learned  2007 Computation and Language


## 6. Generate embeddings via APIM

Uses `text-embedding-3-large` (3,072 dims) through the APIM gateway. Documents are
processed in batches of 100. Exponential backoff handles transient rate-limit errors.

> **Token estimate**: ~6M tokens for 3,000 titles + abstracts. This batch consumes
> from the `IQ_GATEWAY_KEY` subscription quota. Use a dedicated APIM subscription
> (e.g. `foundry-gateway-iq`) to avoid competing with team quotas.

In [6]:
BATCH_SIZE  = 100
MAX_RETRIES = 5
BASE_DELAY  = 2


def embed_texts(texts: list) -> list:
    """Embed a batch of texts via APIM with exponential backoff."""
    for attempt in range(MAX_RETRIES):
        try:
            resp = embed_client.embeddings.create(input=texts, model=embedding_model)
            return [item.embedding for item in resp.data]
        except Exception as exc:
            if attempt == MAX_RETRIES - 1:
                raise
            delay = BASE_DELAY * (2 ** attempt)
            print(f'  Retry {attempt + 1}/{MAX_RETRIES - 1} after error: {exc}. Waiting {delay}s...')
            time.sleep(delay)


title_vectors    = []
abstract_vectors = []
total_batches    = (len(df) + BATCH_SIZE - 1) // BATCH_SIZE

for i in range(0, len(df), BATCH_SIZE):
    batch     = df.iloc[i: i + BATCH_SIZE]
    batch_num = i // BATCH_SIZE + 1

    title_vectors.extend(embed_texts(batch['title'].tolist()))
    abstract_vectors.extend(embed_texts(batch['abstract'].tolist()))

    print(f'  Batch {batch_num:3d}/{total_batches} — {min(i + BATCH_SIZE, len(df)):4d} docs embedded')

df['titleVector']    = title_vectors
df['abstractVector'] = abstract_vectors
print(f'\nDone. Vector dims: {len(df["titleVector"].iloc[0])}')

  Batch   1/30 —  100 docs embedded
  Batch   2/30 —  200 docs embedded
  Batch   3/30 —  300 docs embedded
  Batch   4/30 —  400 docs embedded
  Batch   5/30 —  500 docs embedded
  Batch   6/30 —  600 docs embedded
  Batch   7/30 —  700 docs embedded
  Batch   8/30 —  800 docs embedded
  Batch   9/30 —  900 docs embedded
  Batch  10/30 — 1000 docs embedded
  Batch  11/30 — 1100 docs embedded
  Batch  12/30 — 1200 docs embedded
  Batch  13/30 — 1300 docs embedded
  Batch  14/30 — 1400 docs embedded
  Batch  15/30 — 1500 docs embedded
  Batch  16/30 — 1600 docs embedded
  Batch  17/30 — 1700 docs embedded
  Batch  18/30 — 1800 docs embedded
  Batch  19/30 — 1900 docs embedded
  Batch  20/30 — 2000 docs embedded
  Batch  21/30 — 2100 docs embedded
  Batch  22/30 — 2200 docs embedded
  Batch  23/30 — 2300 docs embedded
  Batch  24/30 — 2400 docs embedded
  Batch  25/30 — 2500 docs embedded
  Batch  26/30 — 2600 docs embedded
  Batch  27/30 — 2700 docs embedded
  Batch  28/30 — 2800 docs e

## 7. Upload documents

Documents are uploaded in batches of 100. Each document includes all text fields,
the `group_ids` security field, and the pre-computed embedding vectors.

**`group_ids` assignment** — in this lab, papers from before 2010 are tagged
`['archive']` and papers from 2010 onwards are tagged `['public', 'archive']`.
This creates a realistic access-control scenario for the security trimming
demonstration in 10-04-search-patterns and 10-03-knowledge-base-setup.

In [7]:
BATCH_SIZE = 100

docs = [
    {
        'id':             str(idx),
        'title':          row['title'],
        'abstract':       row['abstract'],
        'year':           int(row['year']),
        'categories':     row['categories'],
        # Security groups: older papers require 'archive' membership; newer papers
        # are accessible to both 'public' and 'archive' members.
        'group_ids':      ['public', 'archive'] if int(row['year']) >= 2010 else ['archive'],
        'titleVector':    row['titleVector'],
        'abstractVector': row['abstractVector'],
    }
    for idx, row in df.iterrows()
]

total_batches = (len(docs) + BATCH_SIZE - 1) // BATCH_SIZE
for i in range(0, len(docs), BATCH_SIZE):
    batch     = docs[i: i + BATCH_SIZE]
    batch_num = i // BATCH_SIZE + 1
    search_client.upload_documents(batch)
    print(f'  Batch {batch_num:3d}/{total_batches} — {min(i + BATCH_SIZE, len(docs)):4d} docs uploaded')

print(f'\nTotal: {len(docs)} documents uploaded')

  Batch   1/30 —  100 docs uploaded
  Batch   2/30 —  200 docs uploaded
  Batch   3/30 —  300 docs uploaded
  Batch   4/30 —  400 docs uploaded
  Batch   5/30 —  500 docs uploaded
  Batch   6/30 —  600 docs uploaded
  Batch   7/30 —  700 docs uploaded
  Batch   8/30 —  800 docs uploaded
  Batch   9/30 —  900 docs uploaded
  Batch  10/30 — 1000 docs uploaded
  Batch  11/30 — 1100 docs uploaded
  Batch  12/30 — 1200 docs uploaded
  Batch  13/30 — 1300 docs uploaded
  Batch  14/30 — 1400 docs uploaded
  Batch  15/30 — 1500 docs uploaded
  Batch  16/30 — 1600 docs uploaded
  Batch  17/30 — 1700 docs uploaded
  Batch  18/30 — 1800 docs uploaded
  Batch  19/30 — 1900 docs uploaded
  Batch  20/30 — 2000 docs uploaded
  Batch  21/30 — 2100 docs uploaded
  Batch  22/30 — 2200 docs uploaded
  Batch  23/30 — 2300 docs uploaded
  Batch  24/30 — 2400 docs uploaded
  Batch  25/30 — 2500 docs uploaded
  Batch  26/30 — 2600 docs uploaded
  Batch  27/30 — 2700 docs uploaded
  Batch  28/30 — 2800 docs u

## 8. Verify document count

In [8]:
time.sleep(3)  # brief pause for index freshness

count = search_client.get_document_count()
print(f'Document count: {count}')
print()
print('Index is ready. Open 10-04-search-patterns.ipynb for the retrieval demonstrations.')

Document count: 3000

Index is ready. Open 09-04-search-patterns.ipynb for the retrieval demonstrations.


## 9. Cleanup *(optional)*

Deletes the index. Run this only when you have finished all four notebooks in the series.

In [9]:
# index_client.delete_index(INDEX_NAME)
# print(f"Index '{INDEX_NAME}' deleted.")